In [1]:
# ================================================
# 1. Imports
# ================================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [2]:
# ================================================
# 2. Load datasets (train + eval)
# ================================================
train_df = pd.read_csv(r"C:\Users\Aashi\Regression_ML_EndtoEnd\data\processed\feature_engineered_train.csv")
eval_df  = pd.read_csv(r"C:\Users\Aashi\Regression_ML_EndtoEnd\data\processed\feature_engineered_eval.csv")

In [12]:
'''
# ================================================
# 3. Drop high VIF features (both train + eval)
# ================================================
high_vif_features = [
    "median_sale_price" #highest correlation to 'price' => data leakage
]
train_df.drop(columns=high_vif_features, inplace=True)
eval_df.drop(columns=high_vif_features, inplace=True)
'''

'\n# ================================================\n# 3. Drop high VIF features (both train + eval)\n# ================================================\nhigh_vif_features = [\n    "median_sale_price" #highest correlation to \'price\' => data leakage\n]\ntrain_df.drop(columns=high_vif_features, inplace=True)\neval_df.drop(columns=high_vif_features, inplace=True)\n'

In [3]:
# ================================================
# 4. Define target & features
# ================================================
target = "price"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_eval = eval_df.drop(columns=[target])
y_eval = eval_df[target]

In [4]:
# ================================================
# 5. Standardization (fit on train, transform eval)
# ================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled  = scaler.transform(X_eval)

In [5]:
# ================================================
# 6. Train & Evaluate Models
# ================================================

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_eval_scaled)

print("Linear Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lr))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lr)))
print(" R²:", r2_score(y_eval, y_pred_lr))

Linear Regression:
 MAE: 49686.66429420353
 RMSE: 132519.72111357044
 R²: 0.8642871228781143


In [8]:
from sklearn.linear_model import RidgeCV
import numpy as np

# --- Ridge Regression with Cross Validation ---
alphas = [0.01, 0.1, 1, 10, 100, 1000, 10000]

ridge = RidgeCV(alphas=alphas, cv=5)
ridge.fit(X_train_scaled, y_train)

print("Best Ridge Alpha:", ridge.alpha_)

y_pred_ridge = ridge.predict(X_eval_scaled)

print("\nRidge Regression:")
print(" MAE:  ", mean_absolute_error(y_eval, y_pred_ridge))
print(" RMSE: ", np.sqrt(mean_squared_error(y_eval, y_pred_ridge)))
print(" R²:   ", r2_score(y_eval, y_pred_ridge))

Best Ridge Alpha: 0.01

Ridge Regression:
 MAE:   49686.66219145386
 RMSE:  132519.82756038936
 R²:    0.8642869048545667


In [10]:
# --- Lasso Regression ---
'''
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_eval_scaled)

print("\nLasso Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lasso))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lasso)))
print(" R²:", r2_score(y_eval, y_pred_lasso)) '''

from sklearn.linear_model import LassoCV

alphas = [0.1, 1, 10, 100]   

lasso = LassoCV(
    alphas=alphas, 
    cv=3,          
    max_iter=1000,  # less iterations
    n_jobs=-1       # use all CPU cores → faster!
)
lasso.fit(X_train_scaled, y_train)

print("Best Lasso Alpha:", lasso.alpha_)

y_pred_lasso = lasso.predict(X_eval_scaled)

print("\nLasso Regression:")
print(" MAE:  ", mean_absolute_error(y_eval, y_pred_lasso))
print(" RMSE: ", np.sqrt(mean_squared_error(y_eval, y_pred_lasso)))
print(" R²:   ", r2_score(y_eval, y_pred_lasso))

c:\Users\Aashi\Regression_ML_EndtoEnd\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.801e+14, tolerance: 1.945e+12
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Aashi\Regression_ML_EndtoEnd\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.154e+14, tolerance: 1.945e+12
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Aashi\Regression_ML_EndtoEnd\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or c

Best Lasso Alpha: 100.0

Lasso Regression:
 MAE:   49597.79780904823
 RMSE:  133490.92761616028
 R²:    0.8622906168077626


In [13]:
# --- ElasticNet ---

elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
y_pred_elastic = elastic.predict(X_eval_scaled)

print("\nElasticNet Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_elastic))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_elastic)))
print(" R²:", r2_score(y_eval, y_pred_elastic))    

'''

from sklearn.linear_model import ElasticNetCV

# --- ElasticNet with Cross Validation ---
alphas    = [0.1, 1, 10, 100]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]  # 0=Ridge, 1=Lasso

elastic = ElasticNetCV(
    alphas=alphas,
    l1_ratio=l1_ratios,
    cv=3,
    max_iter=1000,
    n_jobs=-1        # use all CPU cores → faster!
)
elastic.fit(X_train_scaled, y_train)

print("Best Alpha:",    elastic.alpha_)
print("Best L1 Ratio:", elastic.l1_ratio_)

y_pred_elastic = elastic.predict(X_eval_scaled)

print("\nElasticNet Regression:")
print(" MAE:  ", mean_absolute_error(y_eval, y_pred_elastic))
print(" RMSE: ", np.sqrt(mean_squared_error(y_eval, y_pred_elastic)))
print(" R²:   ", r2_score(y_eval, y_pred_elastic))   '''


ElasticNet Regression:
 MAE: 50377.70675824147
 RMSE: 126218.16377612248
 R²: 0.876887047500215


'\n\nfrom sklearn.linear_model import ElasticNetCV\n\n# --- ElasticNet with Cross Validation ---\nalphas    = [0.1, 1, 10, 100]\nl1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]  # 0=Ridge, 1=Lasso\n\nelastic = ElasticNetCV(\n    alphas=alphas,\n    l1_ratio=l1_ratios,\n    cv=3,\n    max_iter=1000,\n    n_jobs=-1        # use all CPU cores → faster!\n)\nelastic.fit(X_train_scaled, y_train)\n\nprint("Best Alpha:",    elastic.alpha_)\nprint("Best L1 Ratio:", elastic.l1_ratio_)\n\ny_pred_elastic = elastic.predict(X_eval_scaled)\n\nprint("\nElasticNet Regression:")\nprint(" MAE:  ", mean_absolute_error(y_eval, y_pred_elastic))\nprint(" RMSE: ", np.sqrt(mean_squared_error(y_eval, y_pred_elastic)))\nprint(" R²:   ", r2_score(y_eval, y_pred_elastic))   '